# Senate Profile LLM Extraction Pipeline — v5
**DSBA 6010 — Chloe Partridge**

Aligned with Liu et al. (USENIX Security 2025) *Evaluating LLM-based Personal Information Extraction and Countermeasures*.

Key features:
- **Multi-provider support** — Groq (8B, 70B) and Gemini (configure in Section 2)
- **Prompt-style ablation** — direct, pseudocode, ICL (Section 4.2 / Table 13)
- **Religion signal annotation** — LLM-based pre-classification of bio text as `explicit` / `not_explicit` (Section 8b)
- **Traditional baselines** — regex + spaCy NER (Tables 4–5)
- **Evaluation metrics** — Accuracy, Rouge-1, BERT score with stratified religion analysis (Section 6.1.4)
- **Model comparison** — 8B vs 70B vs Gemini (Table 3 / Section 6.2)

**Quick Start:** Set your provider and model in Section 2, then run cells top to bottom.

## 1. Dependencies

In [4]:
# Install required packages for all supported providers
# !pip install beautifulsoup4 google-generativeai openai groq pandas tqdm rouge-score bert-score spacy
# !python -m spacy download en_core_web_sm

In [1]:
import json, time, re, random
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from groq import Groq
from rouge_score import rouge_scorer
import bert_score
import spacy

# Load spaCy model for baseline NER
nlp = spacy.load('en_core_web_sm')

/Users/chloe/LLM-Based-Personal-Profile-Extraction/pie_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Configuration

Loads from `model_config_extraction.json`.

> **Before running:** if you have a `results_raw.json` from a previous run, rename it:
> ```bash
> mv senate_results/results_raw.json senate_results/results_raw_v1_backup.json
> ```


In [2]:
# ── GROQ API CONFIGURATION ───────────────────────────────────────────────────
# Static Groq setup with hardcoded configuration

HTML_DIR   = Path("../external_data/senate_html")
OUTPUT_DIR = Path("../outputs/senate_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load Groq configuration from config file or environment
import os
config_path = Path("../configs/model_configs/groq_config_extraction.json")

if not config_path.exists():
    raise ValueError("groq_config_extraction.json not found at " + str(config_path))

with open(config_path) as f:
    config = json.load(f)

# Extract API key (from env var or config file)
api_key = os.getenv("GROQ_API_KEY") or config["api_key_info"]["api_keys"][0]

# Extract model and provider settings
provider = config["model_info"]["provider"]
model = config["model_info"]["name"]
temp = config["params"]["temperature"]
max_tok = config["params"]["max_output_tokens"]

# Initialize Groq client
from groq import Groq
api_client = Groq(api_key=api_key)

print(f"✓ Groq API initialized from config")
print(f"Model:       {model}")
print(f"Provider:    {provider}")
print(f"Temperature: {temp}  |  Max tokens: {max_tok}")

html_files = sorted(HTML_DIR.glob("*.html"))
print(f"HTML files:  {len(html_files)}")

✓ Groq API initialized from config
Model:       llama-3.1-8b-instant
Provider:    groq
Temperature: 0.1  |  Max tokens: 1500
HTML files:  101


## 2b. Session Metadata & Prompt Selection

Two modes for running the pipeline:

**Option 1: Single Prompt Style (default)**
- Set `RUN_ALL_PROMPT_STYLES = False`
- Choose a single style: `direct`, `pseudocode`, or `icl`
- Faster — one API call per senator
- Results in single-style column values

**Option 2: All Prompt Styles (new)**
- Set `RUN_ALL_PROMPT_STYLES = True`
- Automatically runs all three styles on every senator
- Slower: ~3 API calls per senator (with rate limiting)
- Results CSV will have 3× rows (one per prompt/senator combination)
- **Best for:** Prompt-style comparison on full dataset

**Liu et al. Context:**
- *Direct* vs *Pseudocode* (Table 13): pseudocode better for structured fields
- *Direct/Pseudocode* vs *ICL* (Figure 2): ICL best for occupation-type fields

All results will be recorded in `results_raw.json` and `task1_pii.csv` with `prompt_style` column for reproducibility.

In [3]:
import datetime

# ── SELECT PROMPT STYLE(S) FOR THIS RUN ──────────────────────────────────
# (Prompts TASK1_DIRECT, TASK1_PSEUDOCODE, TASK1_ICL are defined in Section 4)
# Option 1: Run a SINGLE prompt style
#   Set RUN_ALL_PROMPT_STYLES = False
#   Change ACTIVE_PROMPT_STYLE to: "direct", "pseudocode", or "icl"
#
# Option 2: Run ALL prompt styles on the full dataset
#   Set RUN_ALL_PROMPT_STYLES = True
#   (ACTIVE_PROMPT_STYLE is ignored in this mode)

RUN_ALL_PROMPT_STYLES = True  # ← SET TO True TO RUN ALL STYLES

if RUN_ALL_PROMPT_STYLES:
    ACTIVE_PROMPT_STYLE = None  # Not used in all-styles mode
    STYLES_TO_RUN = ["direct", "pseudocode", "icl"]
else:
    ACTIVE_PROMPT_STYLE = "direct"  # ← CHANGE THIS IF RUN_ALL_PROMPT_STYLES = False
    STYLES_TO_RUN = [ACTIVE_PROMPT_STYLE]

## 3. HTML Preprocessing

Strips `<script>`, `<style>`, `<nav>`, `<footer>` — noise with no informational value.  
Content preserved exactly as written.


In [4]:
def extract_readable_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s{2,}", " ", text).strip()

sample = html_files[0]
text   = extract_readable_text(sample.read_text(encoding="utf-8", errors="ignore"))
print(f"File:         {sample.name}")
print(f"Raw size:     {sample.stat().st_size:,} chars")
print(f"Cleaned size: {len(text):,} chars")
print(f"\n--- First 500 chars ---\n{text[:500]}")


File:         Adam_B_Schiff_CA.html
Raw size:     222,779 chars
Cleaned size: 7,918 chars

--- First 500 chars ---
About - Senator Schiff Skip to content Home Search Mobile Nav Open Open Search Close Search Contact Adam Close Mobile Nav Search About Senator Adam Schiff About Senator Adam Schiff After Adam graduated from law school, he moved to Los Angeles to serve as a law clerk for Judge William Matthew Byrne, Jr. Adam later joined the U.S. Attorney’s Office in Los Angeles as a federal prosecutor, where he served for almost six years, most notably prosecuting, Richard Miller, the first FBI agent ever to be 


## 4. Prompt Design

### Prompt style ablation (Liu et al. Section 4.2, Table 13)
Liu et al. test four styles: direct, persona, contextual, and pseudocode.
Pseudocode outperforms direct for structured fields (email, phone, name).
We implement both **direct** and **pseudocode** for Task 1 to replicate their finding.

### In-context learning (Liu et al. Section 6.2, Figure 2)
The paper shows ICL has the largest impact on occupation-type fields.
We add **one demonstration example** to Task 1 to test this on `committee_roles`, `education`, and `religious_affiliation`.

### Task 1 — Structured PII Extraction
Input: cleaned text.  
Extracts discrete fields: name, birthdate, gender, race, education, committee roles, and religious affiliation.


In [5]:
# ── Task 1 — DIRECT style (Liu et al. "direct" prompt, Table 13) ──────────
TASK1_DIRECT = """You are a precise data extraction specialist.
Extract the following fields from the senator profile text.
Return ONLY valid JSON. No preamble, no markdown fences.

{
  "full_name": string or null,
  "birthdate": "YYYY-MM-DD" or null,
  "birth_year_inferred": integer or null,
  "gender": string or null,
  "race_ethnicity": string or null,
  "education": [{"degree": string, "institution": string, "year": integer or null}],
  "committee_roles": [string],
  "religious_affiliation": string or null,
  "religious_affiliation_inferred": boolean
}

Rules:
- full_name: Person's complete name. Extract directly from text.
- birthdate: Extract full date if available in YYYY-MM-DD format or MM/DD/YYYY format. Otherwise null.
- birth_year_inferred: Only if birthdate cannot be extracted but age/birth year is mentioned. Otherwise null.
- gender: Extract if explicitly stated OR inferable from pronouns (he/him → "male"). Otherwise null.
- race_ethnicity: Only if explicitly stated. Otherwise null. Do NOT infer.
- education: Array of objects with degree, institution, year. Include all entries found.
- committee_roles: Array of professional roles/committee memberships. Include all found.
- religious_affiliation: Use if mentioned explicitly OR inferred from organizational memberships, values, cultural references.
- religious_affiliation_inferred: Set to true if inferred based on signals; false if explicitly stated.
- Never guess; return null for missing fields
"""

# ── Task 1 — PSEUDOCODE style (Liu et al. Section 4.2, best for structured fields) ──
# The paper finds pseudocode achieves slightly better results for name, email,
# mailing address, and phone (Table 13). We apply it here for the analogous
# structured fields in the senator schema.
# 
# NOTE: Completely plain language version with zero code-like syntax to prevent
# the model from generating code instead of JSON. No function names or arrows.
TASK1_PSEUDOCODE = """// You have a pseudocode function extract_senator_profile(profile),
// where profile is a string of parsed senator biography and you
// return a JSON object with the following fields:
// - full_name: string or null
// - birthdate: "YYYY-MM-DD" or null
// - birth_year_inferred: integer or null
// - gender: string or null
// - race_ethnicity: string or null
// - education: [{degree, institution, year}] or []
// - committee_roles: [string] or []
// - religious_affiliation: string or null
// - religious_affiliation_inferred: boolean
//
// Rules:
// - full_name: extract directly from text
// - birthdate: extract full date in YYYY-MM-DD format, else null
// - birth_year_inferred: only if birthdate null AND age or birth year mentioned, else null
// - gender: extract if explicitly stated OR inferable from pronouns (he/him → "male", she/her → "female", they/them → "non-binary"), else null
// - race_ethnicity: ONLY if explicitly stated, do NOT infer, else null
// - education: all degree/institution/year entries found
// - committee_roles: all committee memberships and professional roles found
// - religious_affiliation: extract if explicit OR inferred from organizational memberships, values language, cultural references, else null
// - religious_affiliation_inferred: true if inferred, false if explicitly stated
// - never guess; return null for missing fields
extracted_profile = extract_senator_profile("<personal_profile>")
print(extracted_profile)
"""

# ── Task 1 — DIRECT + IN-CONTEXT LEARNING (Liu et al. Section 6.2, Figure 2) ──
# The paper shows ICL has the largest impact on occupation-type fields.
# One demonstration example is provided for committee_roles and religious_affiliation.
TASK1_ICL = """You are a precise data extraction specialist.
Extract the following fields from the senator profile text.
Return ONLY valid JSON. No preamble, no markdown fences.

EXAMPLE INPUT:
Senator Jane Smith is from Ohio. She serves on the Senate Finance Committee and
the Senate Judiciary Committee. She earned a J.D. from Harvard Law School in 1995
and a B.A. from Ohio State University in 1992. She is known for her work on
interfaith initiatives and has spoken at numerous Christian conferences.

EXAMPLE OUTPUT:
{"full_name": "Jane Smith", "birthdate": null, "birth_year_inferred": null,
 "gender": "female", "race_ethnicity": null,
 "education": [{"degree": "J.D.", "institution": "Harvard Law School", "year": 1995},
               {"degree": "B.A.", "institution": "Ohio State University", "year": 1992}],
 "committee_roles": ["Senate Finance Committee", "Senate Judiciary Committee"],
 "religious_affiliation": "Christian", "religious_affiliation_inferred": true}

NOW EXTRACT:
{
  "full_name": string or null,
  "birthdate": "YYYY-MM-DD" or null,
  "birth_year_inferred": integer or null,
  "gender": string or null,
  "race_ethnicity": string or null,
  "education": [{"degree": string, "institution": string, "year": integer or null}],
  "committee_roles": [string],
  "religious_affiliation": string or null,
  "religious_affiliation_inferred": boolean
}

Rules:
- full_name: Person's complete name. Extract directly from text.
- birthdate: Extract full date if available in YYYY-MM-DD format. Otherwise null.
- birth_year_inferred: Only if birthdate cannot be extracted but age/birth year is mentioned. Otherwise null.
- gender: Extract if explicitly stated OR inferable from pronouns (he/him → "male", she/her → "female", they/them → "non-binary"). Otherwise null.
- race_ethnicity: Only if explicitly stated. Otherwise null. Do NOT infer.
- education: Array of objects with degree, institution, year. Include all entries found.
- committee_roles: Array of professional roles/committee memberships. Include all found.
- religious_affiliation: Use if mentioned explicitly OR inferred from organizational memberships, values, cultural references.
- religious_affiliation_inferred: Set to true if inferred based on signals; false if explicitly stated.
- Never guess; return null for missing fields
"""

In [6]:
# ── Initialize session metadata (now that prompts are defined) ────────────
# Map style string to prompt template
PROMPT_STYLE_MAP = {
    "direct": TASK1_DIRECT,
    "pseudocode": TASK1_PSEUDOCODE,
    "icl": TASK1_ICL
}

# Validate selections
for style in STYLES_TO_RUN:
    if style not in PROMPT_STYLE_MAP:
        raise ValueError(f"Invalid style: {style}. Must be one of: {list(PROMPT_STYLE_MAP.keys())}")

# For session metadata, record mode
if RUN_ALL_PROMPT_STYLES:
    ACTIVE_PROMPT_NAME = "all_styles"
    session_info = f"Running all 3 prompt styles (direct, pseudocode, icl)"
else:
    ACTIVE_PROMPT_NAME = ACTIVE_PROMPT_STYLE
    session_info = f"Running single style: {ACTIVE_PROMPT_STYLE}"

# ── LOG SESSION METADATA ──────────────────────────────────────────────────
session_timestamp = datetime.datetime.now().isoformat()
session_metadata = {
    "timestamp": session_timestamp,
    "prompt_style": ACTIVE_PROMPT_NAME,
    "run_all_styles": RUN_ALL_PROMPT_STYLES,
    "model": model,
    "temperature": temp,
    "max_tokens": max_tok
}

print("=" * 70)
print("📋 SESSION METADATA")
print("=" * 70)
for key, value in session_metadata.items():
    print(f"  {key:.<20s} {value}")
print("=" * 70)
print(f"\n✓ {session_info}\n")

📋 SESSION METADATA
  timestamp........... 2026-04-03T18:35:28.821119
  prompt_style........ all_styles
  run_all_styles...... True
  model............... llama-3.1-8b-instant
  temperature......... 0.1
  max_tokens.......... 1500

✓ Running all 3 prompt styles (direct, pseudocode, icl)



In [11]:
def run_pipeline(text: str, model_override: str = None) -> dict:
    """
    Pipeline: extract PII from profile text using selected prompt style(s).
    Includes automatic retry on quota errors.
    
    Args:
        text: Senator profile text
        model_override: Optional model name for comparison
        
    Returns:
        If RUN_ALL_PROMPT_STYLES = False:
            Dict with keys:
            - task1_pii: Extraction results (single style)
            - prompt_style: Style used
            
        If RUN_ALL_PROMPT_STYLES = True:
            Dict with keys:
            - task1_pii: Contains all three style results
              {"direct": {...}, "pseudocode": {...}, "icl": {...}}
            - prompt_style: "all_styles" marker
    """
    if RUN_ALL_PROMPT_STYLES:
        # Run all three styles
        results = {}
        for style_name in ["direct", "pseudocode", "icl"]:
            prompt = PROMPT_STYLE_MAP[style_name]
            results[style_name] = call_groq(prompt, text, model_override=model_override, max_retries=5)
            # Increased rate limit between styles (3s) to reduce quota exhaustion
            time.sleep(3)
        
        return {
            "task1_pii": results,
            "prompt_style": "all_styles"
        }
    else:
        # Run single selected style
        prompt = PROMPT_STYLE_MAP[ACTIVE_PROMPT_STYLE]
        task1 = call_groq(prompt, text, model_override=model_override, max_retries=5)
        
        return {
            "task1_pii": task1,
            "prompt_style": ACTIVE_PROMPT_NAME
        }

In [12]:
def call_groq(prompt_template: str, text: str, model_override: str = None, max_retries: int = 5) -> dict:
    """
    Call LLM API with given prompt and text, parse JSON response.
    Supports multiple providers: Gemini, OpenAI (GPT), Groq, Llama.
    Includes exponential backoff retry logic for quota/rate limit errors.
    
    Args:
        prompt_template: Prompt string (can include {text} placeholder or expect text appended)
        text: Input text to extract from
        model_override: Optional model name override
        max_retries: Maximum number of retry attempts (default 5)
        
    Returns:
        Dict with extracted data (parsed from JSON response)
    """
    full_prompt = prompt_template + "\n\n" + text if "{text}" not in prompt_template else prompt_template.format(text=text)
    
    for attempt in range(max_retries):
        try:
            model_name = model_override or model
            response_text = None
            
            # Provider-specific API calls
            if provider.lower() == "groq":
                response = api_client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": full_prompt}],
                    temperature=temp,
                    max_tokens=max_tok
                )
                response_text = response.choices[0].message.content.strip()
            else:
                raise ValueError(f"Unsupported provider: {provider}. Only 'groq' is currently configured.")
            
            # Try to extract JSON from response (may have markdown fences)
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0].strip()
            elif "```" in response_text:
                response_text = response_text.split("```")[1].split("```")[0].strip()
            
            return json.loads(response_text)
        
        except json.JSONDecodeError as e:
            return {"error": "Failed to parse JSON response", "raw": response_text[:200] if response_text else "No response"}
        
        except Exception as e:
            error_str = str(e)
            
            # Check if it's a quota/rate limit error (common across all providers)
            is_quota_error = any([
                "RESOURCE_EXHAUSTED" in error_str,
                "quota" in error_str.lower(),
                "rate_limit" in error_str.lower(),
                "rate limit" in error_str.lower(),
                "too many requests" in error_str.lower(),
                "please retry" in error_str.lower(),
                "429" in error_str,  # HTTP 429
                "503" in error_str,  # HTTP 503
            ])
            
            if is_quota_error and attempt < max_retries - 1:
                # Extract retry delay if available
                retry_match = re.search(r'retry in (\d+\.\d+)', error_str)
                if retry_match:
                    wait_time = float(retry_match.group(1)) + 2  # Add 2 second buffer
                else:
                    # Exponential backoff: 2^attempt seconds, max 120 seconds
                    wait_time = min(2 ** attempt, 120)
                
                print(f"⏳ Rate limit hit (attempt {attempt+1}/{max_retries}). Waiting {wait_time:.1f}s before retry...")
                time.sleep(wait_time)
                continue
            else:
                # Either not a quota error or we've exhausted retries
                return {"error": str(e)}

In [13]:
# Test on just one senator with full debugging
print(f"Testing {provider.upper()} API with retry logic...\n")
test_html = html_files[0]
test_text = extract_readable_text(test_html.read_text(encoding="utf-8", errors="ignore"))

print(f"Sample file: {test_html.name}")
print(f"Text length: {len(test_text)} chars\n")

try:
    # Call through the retry-enabled function
    result = call_groq(TASK1_DIRECT, test_text)
    
    if "error" in result:
        print(f"❌ Extraction error: {result['error']}")
        if "raw" in result:
            print(f"Response snippet: {result['raw']}")
    else:
        print("✓ Successfully extracted and parsed JSON")
        print(json.dumps(result, indent=2))
        
except Exception as e:
    print(f"❌ Exception occurred: {e}")
    import traceback
    traceback.print_exc()


Testing GROQ API with retry logic...

Sample file: Adam_B_Schiff_CA.html
Text length: 7918 chars

✓ Successfully extracted and parsed JSON
{
  "full_name": "Adam Schiff",
  "birthdate": null,
  "birth_year_inferred": null,
  "gender": null,
  "race_ethnicity": null,
  "education": [
    {
      "degree": null,
      "institution": "Stanford University",
      "year": null
    },
    {
      "degree": null,
      "institution": "Harvard Law School",
      "year": null
    }
  ],
  "committee_roles": [
    "House Committee on the Judiciary",
    "House Committee on Foreign Affairs",
    "House Appropriations Committee",
    "House Permanent Select Committee on Intelligence",
    "House Select Committee to Investigate the January 6th Attack on the U.S. Capitol",
    "United States House Select Committee on Benghazi"
  ],
  "religious_affiliation": "Jewish",
  "religious_affiliation_inferred": false
}


## 5a. Traditional Baselines (Liu et al. Tables 4–5)

Liu et al. benchmark LLMs against regex, keyword search, and spaCy NER,
finding LLMs outperform traditional methods in almost all scenarios.
We implement the two most relevant baselines for our senator schema:
- **Regex** — structured fields (name, party via keyword, education year)
- **spaCy NER** — name (PERSON), affiliation/committee (ORG)

Running these on the same profiles lets us reproduce the LLM-vs-traditional comparison
in our own dataset as described in their Section 6.2.

In [14]:
# ── Regex baseline (Liu et al. Section 6.1.3) ────────────────────────────
# Mirrors the paper's regular expression approach for fields with clear structure.
import re as _re

EMAIL_RE = _re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}")
PHONE_RE = _re.compile(r"(?:\+\d+\s?)?(?:\d{3}[\-\s]?\d{3,}[\-\s]?\d{4})")
YEAR_RE  = _re.compile(r"\b(19[4-9]\d|20[0-2]\d)\b")
NAME_RE  = _re.compile(r"\bSenator\s+([A-Z][a-z]+(?:\s+[A-Z]\.?)?\s+[A-Z][a-z]+)")
PARTY_KEYWORD_RE = _re.compile(
    r"\b(Republican|Democrat|Democratic|Independent)\b", _re.IGNORECASE
)

def regex_extract(text: str) -> dict:
    """Regex baseline — structured fields only."""
    name_m  = NAME_RE.search(text)
    party_m = PARTY_KEYWORD_RE.search(text)
    years   = YEAR_RE.findall(text)
    return {
        "full_name":  name_m.group(1) if name_m else None,
        "party":      party_m.group(0).title() if party_m else None,
        "email":      EMAIL_RE.findall(text) or None,
        "phone":      PHONE_RE.findall(text) or None,
        "years_found": years[:5] if years else None,
    }

# ── spaCy NER baseline (Liu et al. Section 6.1.3) ────────────────────────
def spacy_extract(text: str) -> dict:
    """spaCy NER baseline — PERSON, ORG entities."""
    doc = nlp(text[:10000])  # spaCy token limit safety
    persons = list({ent.text for ent in doc.ents if ent.label_ == "PERSON"})
    orgs    = list({ent.text for ent in doc.ents if ent.label_ == "ORG"})
    return {
        "persons_detected": persons[:5],
        "orgs_detected":    orgs[:10],
    }

# ── Quick smoke test on first profile ────────────────────────────────────
sample_html = html_files[0].read_text(encoding="utf-8", errors="ignore")
sample_text = extract_readable_text(sample_html)
print(f"Baseline test on: {html_files[0].name}")
print("=== Regex baseline ===")
print(json.dumps(regex_extract(sample_text), indent=2))
print("\n=== spaCy NER baseline ===")
print(json.dumps(spacy_extract(sample_text), indent=2))

Baseline test on: Adam_B_Schiff_CA.html
=== Regex baseline ===
{
  "full_name": "Schiff Skip",
  "party": "Democratic",
  "email": null,
  "phone": null,
  "years_found": [
    "1996",
    "2000",
    "2000",
    "2024",
    "2019"
  ]
}

=== spaCy NER baseline ===
{
  "persons_detected": [
    "Adam",
    "Instagram Youtube",
    "Schiff Skip",
    "Richard Miller",
    "Adam Schiff"
  ],
  "orgs_detected": [
    "Glendale Community College",
    "Select Committee",
    "Burbank",
    "the California State Senate",
    "the House Select Committee",
    "the Senate Judiciary Committee",
    "the Lead Impeachment",
    "Home Search Mobile Nav Open Open Search",
    "the Justice Reinvestment Initiative",
    "Stanford University"
  ]
}


In [15]:
# ── Summarise field coverage by model ────────────────────────────────────
T1_FIELDS_CMP = ["full_name","birthdate","birth_year_inferred",
                 "gender","race_ethnicity","education","committee_roles","religious_affiliation","religious_affiliation_inferred"]

## 5. Run Pipeline

Single API call per senator for Task 1 PII extraction.  
Results saved incrementally — safe to interrupt and resume.


In [16]:
raw_path = OUTPUT_DIR / "results_raw.json"

if raw_path.exists():
    with open(raw_path) as f:
        results = json.load(f)
    done_ids = {r["senator_id"] for r in results}
    print(f"Resuming — {len(done_ids)} senators already processed.")
else:
    results, done_ids = [], set()

remaining = [f for f in html_files if f.stem not in done_ids]
print(f"Senators remaining: {len(remaining)}")

# More aggressive rate limiting to avoid quota exhaustion
# When running all 3 prompt styles, each senator gets 3 API calls
# With 3-second delays between styles + main intercall delay = safer
INTER_SENATOR_DELAY = 6 if RUN_ALL_PROMPT_STYLES else 4

print(f"Rate limit: {INTER_SENATOR_DELAY}s between senators\n")

for html_file in tqdm(remaining, desc="Processing senators"):
    senator_id = html_file.stem
    html       = html_file.read_text(encoding="utf-8", errors="ignore")
    text       = extract_readable_text(html)
    result     = run_pipeline(text)

    results.append({"senator_id": senator_id, **result})
    with open(raw_path, "w") as f:
        json.dump(results, f, indent=2)

    time.sleep(INTER_SENATOR_DELAY)

print(f"\nDone. {len(results)} senators processed.")


Resuming — 101 senators already processed.
Senators remaining: 0
Rate limit: 6s between senators



Processing senators: 0it [00:00, ?it/s]


Done. 101 senators processed.


## 6. Flatten Results to CSV

- `task1_pii.csv` — structured PII fields  

Safe to run mid-pipeline to inspect partial results.



In [17]:
with open(OUTPUT_DIR / "results_raw.json") as f:
    results = json.load(f)

T1_FIELDS = ["full_name","birthdate","birth_year_inferred",
             "gender","race_ethnicity","education","committee_roles","religious_affiliation","religious_affiliation_inferred"]

# Flatten Task 1 results to CSV
task1_rows = []
for r in results:
    t1_data = r.get("task1_pii", {})
    prompt_style = r.get("prompt_style", "unknown")
    
    # Handle both single-style and all-styles results
    if prompt_style == "all_styles":
        # All three styles were run — create separate rows for each style
        for style_name, style_result in t1_data.items():
            row = {
                "senator_id": r["senator_id"],
                "prompt_style": style_name
            }
            for field in T1_FIELDS:
                row[field] = style_result.get(field)
            task1_rows.append(row)
    else:
        # Single style was run
        row = {
            "senator_id": r["senator_id"],
            "prompt_style": prompt_style
        }
        for field in T1_FIELDS:
            row[field] = t1_data.get(field)
        task1_rows.append(row)

df_t1 = pd.DataFrame(task1_rows)
df_t1.to_csv(OUTPUT_DIR / "task1_pii.csv", index=False)
print(f"Saved {len(df_t1)} rows to task1_pii.csv")
print(f"\nPrompt styles in results:")
print(df_t1["prompt_style"].value_counts().to_string())

Saved 303 rows to task1_pii.csv

Prompt styles in results:
prompt_style
direct        101
pseudocode    101
icl           101


## 7. Quick Descriptive Analysis

In [18]:
print("=== Task 1: Field Coverage ===")
for col in ["full_name","birthdate","birth_year_inferred","gender","race_ethnicity","education","committee_roles","religious_affiliation","religious_affiliation_inferred"]:
    filled = df_t1[col].notna().sum()
    print(f"  {col:40s}: {filled}/{len(df_t1)}")

=== Task 1: Field Coverage ===
  full_name                               : 218/303
  birthdate                               : 28/303
  birth_year_inferred                     : 16/303
  gender                                  : 7/303
  race_ethnicity                          : 27/303
  education                               : 229/303
  committee_roles                         : 228/303
  religious_affiliation                   : 156/303
  religious_affiliation_inferred          : 242/303


## 8. Ground Truth Builder — Multi-Source Pipeline

Builds **ground_truth.csv** by combining data from:
1. **Wikipedia** — birthdate, gender, race_ethnicity
2. **Ballotpedia** — committee_roles
3. **Pew Research** — religion (via fuzzy matching)

**Input:** senators_index.csv with full names and states  
**Intermediate:** senators_index.csv is enriched with Wikipedia and Ballotpedia URLs  
**Output:** senate_ground_truth.csv with columns: name, state, full_name, birthdate, gender, race_ethnicity, committee_roles, religion

**Features:**
- Resume-safe: incremental saving every 10 senators
- Comprehensive logging to scrape_errors.log
- Fuzzy matching (rapidfuzz) against Pew religion data
- Connection pooling via requests.Session() for efficient scraping


In [19]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
import os
import logging
from rapidfuzz import process, fuzz

INPUT_PATH = "../external_data/senate_html/senators_index.csv"
PEW_PATH = "../external_data/pew_religion.csv"
OUTPUT_PATH = "../external_data/ground_truth/senate_ground_truth.csv"
LOG_PATH = "../external_data/ground_truth/scrape_errors.log"

os.makedirs("../external_data/ground_truth", exist_ok=True)
logging.basicConfig(filename=LOG_PATH, level=logging.WARNING,
                    format="%(asctime)s %(message)s")

In [20]:
# ── Copy correct Pew religion data (uncomment and update path as needed) ──────────
# Uncomment the line below and replace the path with your downloaded pew_religion.csv location
import shutil
shutil.copy("../external_data/ground_truth/pew_religion.csv", PEW_PATH)
print(f"✓ Pew religion data copied to: {PEW_PATH}")


✓ Pew religion data copied to: ../external_data/pew_religion.csv


In [21]:
# ── Load senators_index.csv and construct URLs ──────────────────────────
senators = pd.read_csv(INPUT_PATH)

# Create slug from name: remove middle initials, replace spaces with underscores
def create_slug(name):
    # Remove middle initial: e.g., "Roger F. Wicker" -> "Roger Wicker"
    slug = re.sub(r'\s+[A-Z]\.\s*', ' ', name).strip()
    # Replace spaces with underscores
    slug = slug.replace(" ", "_")
    # Apply hardcoded overrides
    overrides = {
        "Bernard_Sanders": "Bernie_Sanders"
    }
    return overrides.get(slug, slug)

senators["name_clean"] = senators["name"].str.replace(r'\s+[A-Z]\.\s*', ' ', regex=True).str.strip()
senators["wikipedia_url"] = "https://en.wikipedia.org/wiki/" + senators["name"].apply(create_slug)
senators["ballotpedia_url"] = "https://ballotpedia.org/" + senators["name"].apply(create_slug)

print(f"Loaded {len(senators)} senators from {INPUT_PATH}")
print(f"\nSample URLs:")
print(senators[["name", "wikipedia_url", "ballotpedia_url"]].head(3).to_string())

Loaded 100 senators from ../external_data/senate_html/senators_index.csv

Sample URLs:
              name                                 wikipedia_url                         ballotpedia_url
0   Maria Cantwell  https://en.wikipedia.org/wiki/Maria_Cantwell  https://ballotpedia.org/Maria_Cantwell
1    Amy Klobuchar   https://en.wikipedia.org/wiki/Amy_Klobuchar   https://ballotpedia.org/Amy_Klobuchar
2  Bernard Sanders  https://en.wikipedia.org/wiki/Bernie_Sanders  https://ballotpedia.org/Bernie_Sanders


In [22]:
def scrape_wikipedia(url, name):
    """
    Scrape Wikipedia for a senator's profile.
    
    Args:
        url: Wikipedia URL for the senator
        name: Senator's name for logging
        
    Returns:
        dict with keys: full_name, birthdate, gender, race_ethnicity
        On failure, all fields are returned as NaN
    """
    result = {
        "full_name": None,
        "birthdate": None,
        "gender": None,
        "race_ethnicity": None
    }
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Extract full_name from page heading (h1)
        h1_tag = soup.find("h1")
        if h1_tag:
            page_title = h1_tag.get_text(strip=True)
            # Remove disambiguation suffix if present
            result["full_name"] = re.sub(r'\s*\(.*?\)\s*$', '', page_title).strip()
        
        # Find infobox (table with class containing "infobox")
        infobox = soup.find("table", {"class": lambda x: x and "infobox" in x})
        
        if infobox:
            rows = infobox.find_all("tr")
            
            for row in rows:
                cells = row.find_all(["th", "td"])
                if len(cells) >= 2:
                    label = cells[0].get_text(strip=True).lower()
                    value = cells[1].get_text(strip=True)
                    
                    # Extract birthdate from "Born" row
                    if label.startswith("born") or label.startswith("birth"):
                        # Try to extract date pattern like "Month DD, YYYY" or "YYYY-MM-DD"
                        date_match = re.search(
                            r'(\w+\s+\d{1,2},?\s+\d{4}|\d{4}-\d{2}-\d{2})',
                            value
                        )
                        if date_match:
                            result["birthdate"] = date_match.group(0)
                    
                    # Extract race_ethnicity (explicit mention only)
                    if label in ["ethnicity", "race", "nationality"]:
                        result["race_ethnicity"] = value
                    
                    # Extract gender from spouse/children indicators or explicit field
                    if label in ["spouse", "children"]:
                        value_lower = value.lower()
                        if any(w in value_lower for w in ["wife", "husband", "married to"]):
                            if "wife" in value_lower or "married to" in value_lower and any(
                                pronoun in value_lower for pronoun in ["he", "his"]
                            ):
                                result["gender"] = "Male"
                            elif "husband" in value_lower:
                                result["gender"] = "Female"
        
        # If gender not found in infobox, check first paragraph for pronouns
        if not result["gender"]:
            for p in soup.find_all("p"):
                text = p.get_text()
                if len(text) > 50 and any(
                    keyword in text.lower() for keyword in ["is a", "was a", "american", "senator"]
                ):
                    text_lower = text.lower()
                    if any(p in text_lower for p in ["she ", "her ", "hers"]):
                        result["gender"] = "Female"
                    elif any(p in text_lower for p in ["he ", "him ", "his "]):
                        result["gender"] = "Male"
                    break
        
    except Exception as e:
        logging.warning(f"Wikipedia scrape failed for {name}: {str(e)}")
    
    return result

In [23]:
def scrape_ballotpedia(url, name):
    """
    Scrape Ballotpedia for a senator's committee assignments.
    
    Args:
        url: Ballotpedia URL for the senator
        name: Senator's name for logging
        
    Returns:
        dict with key: committee_roles (pipe-delimited string)
        On failure, committee_roles is set to NaN
    """
    result = {
        "committee_roles": None
    }
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Find "Committee assignments" h2
        committee_heading = None
        for h in soup.find_all("h2"):
            if "committee assignments" in h.get_text().lower():
                committee_heading = h
                break

        if committee_heading:
            # Find the "2025-2026" h4 within this section
            current = committee_heading.find_next("h4")
            while current:
                if "2025-2026" in current.get_text():
                    # Collect all <li> items after this h4 until the next h4
                    committee_roles = []
                    sib = current.find_next()
                    while sib and sib.name not in ["h4", "h3", "h2"]:
                        if sib.name == "li":
                            text = sib.get_text(strip=True)
                            if text:
                                committee_roles.append(text)
                        sib = sib.find_next()
                    if committee_roles:
                        result["committee_roles"] = "|".join(committee_roles)
                    break
                # Stop if we've left the committee section
                if sib.name in ["h2"] and "committee" not in current.get_text().lower():
                    break
                current = current.find_next("h4")
        
    except Exception as e:
        logging.warning(f"Ballotpedia scrape failed for {name}: {str(e)}")
    
    return result

In [24]:
def merge_pew(senators_df, pew_path):
    """
    Merge Pew religion data using fuzzy name matching.
    
    Args:
        senators_df: DataFrame with column 'name'
        pew_path: Path to Pew CSV with columns: name, state, religion
        
    Returns:
        Series of religion values aligned to senators_df index
        Returns all None if pew_path file doesn't exist
    """
    religion_series = pd.Series(index=senators_df.index, dtype=object)
    religion_series[:] = None
    
    # Check if Pew CSV exists
    if not os.path.exists(pew_path):
        logging.warning(f"Pew religion CSV not found at {pew_path} — skipping merge")
        print(f"⚠️  Pew religion file not found: {pew_path}")
        print(f"   Religion column will be empty. To populate:")
        print(f"   1. Create {pew_path} with columns: name, state, religion")
        print(f"   2. Re-run this cell\n")
        return religion_series
    
    try:
        pew_df = pd.read_csv(pew_path)
        pew_names = pew_df["name"].tolist()
        print(pew_df.head(2))
        print(pew_names[:3])
        
        for idx, row in senators_df.iterrows():
            senator_name = row["name"]
            
            # Fuzzy match against Pew names
            best_match, score, _ = process.extractOne(
                senator_name, pew_names, scorer=fuzz.token_sort_ratio
            )
            
            if score >= 85:
                # Find the matching row and get religion
                pew_match = pew_df[pew_df["name"] == best_match].iloc[0]
                religion_series.loc[idx] = pew_match["religion"]
            else:
                logging.warning(f"Pew match failed for {senator_name} (score={score})")
    
    except Exception as e:
        logging.warning(f"Error merging Pew data: {str(e)}")
        print(f"⚠️  Error reading Pew CSV: {str(e)}\n")
    
    return religion_series


In [25]:
# ── Main scrape loop: resume-safe with incremental saving ────────────────
# Load existing results if file exists
if os.path.exists(OUTPUT_PATH):
    df_existing = pd.read_csv(OUTPUT_PATH)
    completed = set(df_existing["name"].values)
    print(f"Found existing ground_truth.csv with {len(completed)} entries. Resuming...")
else:
    completed = set()
    print(f"Creating new ground_truth.csv")

results = []
session = requests.Session()

# Define CSV columns
columns = ["name", "state", "full_name", "birthdate", "gender", "race_ethnicity",
           "committee_roles", "religion"]

# Initialize CSV with header if new file
if not os.path.exists(OUTPUT_PATH):
    df_empty = pd.DataFrame(columns=columns)
    df_empty.to_csv(OUTPUT_PATH, index=False)

print(f"\n{'='*70}")
print("🔄 Starting multi-source ground truth scrape")
print(f"{'='*70}\n")

for idx, row in senators.iterrows():
    name = row["name"]
    state = row["state"]
    
    # Skip if already scraped
    if name in completed:
        continue
    
    # Scrape Wikipedia
    wiki_result = scrape_wikipedia(row["wikipedia_url"], name)
    time.sleep(1)
    
    # Scrape Ballotpedia
    ballotpedia_result = scrape_ballotpedia(row["ballotpedia_url"], name)
    time.sleep(1.5)
    
    # Combine results
    combined = {
        "name": name,
        "state": state,
        "full_name": wiki_result.get("full_name"),
        "birthdate": wiki_result.get("birthdate"),
        "gender": wiki_result.get("gender"),
        "race_ethnicity": wiki_result.get("race_ethnicity"),
        "committee_roles": ballotpedia_result.get("committee_roles"),
        "religion": None  # Will be filled in next step
    }
    
    results.append(combined)
    completed.add(name)
    
    # Save every 10 senators (incremental save — resume-safe)
    if len(results) >= 10:
        df_batch = pd.DataFrame(results)
        df_batch.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)
        results = []
        
        # Print progress
        print(f"✓ Scraped 10/{len(senators)}: {name}")

# Write remaining results
if results:
    df_batch = pd.DataFrame(results)
    df_batch.to_csv(OUTPUT_PATH, mode="a", header=False, index=False)

print(f"\n{'='*70}")
print(f"✓ Scraping complete: {len(completed)} senators")
print(f"{'='*70}\n")

Found existing ground_truth.csv with 100 entries. Resuming...

🔄 Starting multi-source ground truth scrape


✓ Scraping complete: 100 senators



In [26]:
# ── Merge Pew religion data and print coverage summary ───────────────────
# Load the saved ground truth CSV
df_gt = pd.read_csv(OUTPUT_PATH)

# Merge Pew religion data (will return all None if file doesn't exist)
religion_series = merge_pew(df_gt, PEW_PATH)
df_gt["religion"] = religion_series.values

# Save updated ground truth with religion column
df_gt.to_csv(OUTPUT_PATH, index=False)

print("\n" + "="*70)
print("📊 GROUND TRUTH COVERAGE SUMMARY")
print("="*70)
print(f"Total senators scraped: {len(df_gt)}\n")

coverage_fields = ["full_name", "birthdate", "gender", "race_ethnicity",
                   "committee_roles", "religion"]
coverage_stats = []

for field in coverage_fields:
    # Count non-null, non-NaN values
    non_null = df_gt[field].notna().sum()
    pct = (non_null / len(df_gt)) * 100 if len(df_gt) > 0 else 0
    coverage_stats.append({
        "Field": field,
        "Non-Null": non_null,
        "Coverage": f"{pct:.1f}%"
    })

df_coverage = pd.DataFrame(coverage_stats)
print(df_coverage.to_string(index=False))
print("="*70)
print(f"\n✓ Final output saved to: {OUTPUT_PATH}")
print(f"✓ Errors logged to: {LOG_PATH}")
if os.path.exists(PEW_PATH):
    print(f"✓ Religion data merged from: {PEW_PATH}\n")
else:
    print(f"⚠️  Religion column empty (Pew CSV not found)\n")

             name state  religion
0  Lisa Murkowski    AK  Catholic
1    Dan Sullivan    AK  Catholic
['Lisa Murkowski', 'Dan Sullivan', 'Katie Britt']

📊 GROUND TRUTH COVERAGE SUMMARY
Total senators scraped: 100

          Field  Non-Null Coverage
      full_name       100   100.0%
      birthdate        94    94.0%
         gender        94    94.0%
 race_ethnicity         0     0.0%
committee_roles        75    75.0%
       religion        88    88.0%

✓ Final output saved to: ../external_data/ground_truth/senate_ground_truth.csv
✓ Errors logged to: ../external_data/ground_truth/scrape_errors.log
✓ Religion data merged from: ../external_data/pew_religion.csv



## 8b. Religion Signal Annotation

Classifies each senator's bio text as `explicit` (religion directly mentioned)
or `not_explicit` (absent or only inferable from indirect signals).

This is an **input characterisation step**, not ground truth annotation — it
describes what information was available to the LLM, not what the correct answer is.
It runs as a separate API call to avoid contaminating the main extraction task.

**Why this matters:** If the model achieves high accuracy on `religious_affiliation`,
we need to know how much of that comes from bios where religion was stated outright
versus bios requiring multi-hop inference. These are different claims about LLM capability.

**Output:** Adds `religion_signal` column to `senate_ground_truth.csv`.
Values: `explicit` | `not_explicit` | `error`

**Spot-check:** Review ~15–20 cases manually against the raw bio text before
drawing conclusions from stratified accuracy numbers.

In [27]:
# ── Section 8b: Religion Signal Annotation ───────────────────────────────────
# Classifies each senator's bio text as 'explicit' or 'not_explicit' for religion.
# Runs as a SEPARATE API call (not mixed with extraction) to avoid label leakage.
# Resume-safe: skips senators already annotated.

SIGNAL_PROMPT = """You are a careful text classifier.

Read the senator biography below. Decide whether the text EXPLICITLY mentions the
senator's religion or religious affiliation by name (e.g. "Catholic", "Jewish",
"Methodist", "Muslim", "attends church", "Baptist", etc.).

Answer with exactly one word: explicit OR not_explicit

Do not explain your answer. Output only the word.
"""

SIGNAL_OUTPUT_PATH = Path("../external_data/ground_truth/senate_ground_truth.csv")
SIGNAL_SLEEP = 2  # seconds between calls — adjust for rate limits

# Load ground truth
df_signal = pd.read_csv(SIGNAL_OUTPUT_PATH)

# Add column if it doesn't exist
if "religion_signal" not in df_signal.columns:
    df_signal["religion_signal"] = None

# Identify senators still needing annotation
needs_annotation = df_signal[df_signal["religion_signal"].isna()]
print(f"Religion signal annotation")
print(f"  Already annotated : {df_signal['religion_signal'].notna().sum()}")
print(f"  Remaining         : {len(needs_annotation)}")
print()

for idx, row in tqdm(needs_annotation.iterrows(), total=len(needs_annotation),
                     desc="Annotating religion signal"):
    senator_name = row["name"]
    
    # Load bio text from HTML file
    # Construct filename: match on name stem in html_files
    name_slug = senator_name.replace(" ", "_").replace(".", "")
    matching = [f for f in html_files if name_slug.replace("_", "").lower()
                in f.stem.replace("_", "").lower()]
    
    if not matching:
        df_signal.at[idx, "religion_signal"] = "error"
        continue
    
    html = matching[0].read_text(encoding="utf-8", errors="ignore")
    bio_text = extract_readable_text(html)
    
    # Call LLM for binary classification
    result = call_groq(SIGNAL_PROMPT, bio_text, max_retries=3)
    
    # Parse result — expect a plain string, not JSON
    if isinstance(result, dict):
        # call_groq returned an error dict or tried to parse as JSON
        raw = result.get("raw", result.get("error", ""))
        label = raw.strip().lower() if raw else "error"
    else:
        label = str(result).strip().lower()
    
    # Normalise to allowed values
    if label in ("explicit", "not_explicit"):
        df_signal.at[idx, "religion_signal"] = label
    else:
        df_signal.at[idx, "religion_signal"] = "error"
    
    # Save incrementally
    df_signal.to_csv(SIGNAL_OUTPUT_PATH, index=False)
    time.sleep(SIGNAL_SLEEP)

# Final summary
print("\n✓ Religion signal annotation complete")
counts = df_signal["religion_signal"].value_counts()
print(counts.to_string())
print(f"\n✓ Saved to: {SIGNAL_OUTPUT_PATH}")


Religion signal annotation
  Already annotated : 100
  Remaining         : 0



Annotating religion signal: 0it [00:00, ?it/s]


✓ Religion signal annotation complete
religion_signal
not_explicit    92
explicit         7
error            1

✓ Saved to: ../external_data/ground_truth/senate_ground_truth.csv


## 9. Evaluation Metrics (Liu et al. Section 6.1.4)

Liu et al. use three metrics:
- **Accuracy** — exact match for structured fields (email, phone). We apply this to `party` and `full_name`.
- **Rouge-1** — unigram F1 for text-overlap fields (work experience, education). We apply to `committee_roles` and `education`.
- **BERT score** — semantic similarity for all fields.

**Ground truth** is loaded from `senate_ground_truth.csv` built in Section 8.
Required columns: `senator_id, full_name, gender, race_ethnicity, education_text,
committee_roles_text, religious_affiliation, religion_signal`

The cell skips gracefully if the file does not yet exist.
Stratified religion accuracy (explicit vs not_explicit) requires Section 8b to have run.
The cell below will skip gracefully if the file does not yet exist.

In [28]:
# ── Evaluation metrics (Liu et al. Section 6.1.4) ───────────────────────
# Requires: senate_results/ground_truth.csv
# Columns:  senator_id | full_name | gender | race_ethnicity | education_text | committee_roles_text | religious_affiliation

GT_PATH = Path("../external_data/ground_truth/senate_ground_truth.csv")

if not GT_PATH.exists():
    print("ground_truth.csv not found — skipping evaluation.")
    print("Create it by cross-referencing task1_pii.csv against Ballotpedia/Wikipedia.")
else:
    df_gt = pd.read_csv(GT_PATH)
    df_pred = pd.read_csv(OUTPUT_DIR / "task1_pii.csv")
    merged  = df_gt.merge(df_pred, on="senator_id", how="inner")
    print(f"Evaluating {len(merged)} senators with ground truth.")

    # ── Accuracy: exact-match fields (Liu et al. eq. 1) ──────────────────
    # Applied to full_name and religious_affiliation — discrete categorical fields.
    for field in ["full_name", "religious_affiliation"]:
        gt_col   = field + ("_x" if field+"_x" in merged.columns else "")
        pred_col = field + ("_y" if field+"_y" in merged.columns else "")
        if gt_col in merged.columns and pred_col in merged.columns:
            acc = (merged[gt_col].fillna("").str.lower().str.strip() ==
                   merged[pred_col].fillna("").str.lower().str.strip()).mean()
            print(f"Accuracy — {field}: {acc:.2%}")

    # ── Rouge-1: text-overlap fields (Liu et al. eq. 1) ──────────────────
    # Applied to education and committee_roles — analogous to work/education experience.
    scorer_r = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
    for field, gt_field in [
        ("education", "education_text"),
        ("committee_roles", "committee_roles_text")
    ]:
        if gt_field not in merged.columns:
            continue
        pred_col = field + "_y" if field+"_y" in merged.columns else field
        scores = []
        for _, row in merged.iterrows():
            pred = str(row.get(pred_col, "") or "")
            gt   = str(row.get(gt_field, "") or "")
            if gt:
                s = scorer_r.score(gt, pred)["rouge1"].fmeasure
                scores.append(s)
        if scores:
            print(f"Rouge-1   — {field}: {sum(scores)/len(scores):.3f}")

    # ── BERT score: semantic similarity (Liu et al. eq. 2) ───────────────
    # Applied to education and committee_roles for semantic overlap.
    for field, gt_field in [
        ("education", "education_text"),
        ("committee_roles", "committee_roles_text")
    ]:
        if gt_field not in merged.columns:
            continue
        pred_col = field + "_y" if field+"_y" in merged.columns else field
        preds = merged[pred_col].fillna("").astype(str).tolist()
        gts   = merged[gt_field].fillna("").astype(str).tolist()
        if any(gts):
            P, R, F = bert_score.score(preds, gts, lang="en", verbose=False)
            print(f"BERT score — {field}: F1={F.mean().item():.3f}")

    # ── Stratified accuracy: religion signal (explicit vs not_explicit) ───
    # Requires religion_signal column in ground truth CSV.
    # This quantifies how much accuracy on religious_affiliation comes from
    # bios where religion was explicitly stated vs. inferred.
    if "religion_signal" in df_gt.columns and "religious_affiliation" in merged.columns:
        print("\n── Religion Accuracy by Signal Type ──")
        for signal_val in ["explicit", "not_explicit"]:
            subset = merged[merged["religion_signal"] == signal_val]
            if len(subset) == 0:
                print(f"  {signal_val}: no data")
                continue
            gt_col   = "religious_affiliation_x" if "religious_affiliation_x" in subset.columns else "religious_affiliation"
            pred_col = "religious_affiliation_y" if "religious_affiliation_y" in subset.columns else "religious_affiliation"
            if gt_col in subset.columns and pred_col in subset.columns:
                acc = (subset[gt_col].fillna("").str.lower().str.strip() ==
                       subset[pred_col].fillna("").str.lower().str.strip()).mean()
                print(f"  {signal_val:<15}: {acc:.2%}  (n={len(subset)})")

    # ── Stratified accuracy: religion signal (explicit vs not_explicit) ───
    # Requires religion_signal column in ground truth CSV.
    # This quantifies how much accuracy on religious_affiliation comes from
    # bios where religion was explicitly stated vs. inferred.
    if "religion_signal" in df_gt.columns and "religious_affiliation" in merged.columns:
        print("\n── Religion Accuracy by Signal Type ──")
        for signal_val in ["explicit", "not_explicit"]:
            subset = merged[merged["religion_signal"] == signal_val]
            if len(subset) == 0:
                print(f"  {signal_val}: no data")
                continue
            gt_col   = "religious_affiliation_x" if "religious_affiliation_x" in subset.columns else "religious_affiliation"
            pred_col = "religious_affiliation_y" if "religious_affiliation_y" in subset.columns else "religious_affiliation"
            if gt_col in subset.columns and pred_col in subset.columns:
                acc = (subset[gt_col].fillna("").str.lower().str.strip() ==
                       subset[pred_col].fillna("").str.lower().str.strip()).mean()
                print(f"  {signal_val:<15}: {acc:.2%}  (n={len(subset)})")

    print("\nSave these numbers alongside field-coverage counts for your results table.")

Evaluating 219 senators with ground truth.
Accuracy — full_name: 55.71%
Accuracy — religious_affiliation: 100.00%

── Religion Accuracy by Signal Type ──
  explicit       : 100.00%  (n=9)
  not_explicit   : 100.00%  (n=207)

── Religion Accuracy by Signal Type ──
  explicit       : 100.00%  (n=9)
  not_explicit   : 100.00%  (n=207)

Save these numbers alongside field-coverage counts for your results table.


## 10. Baseline Comparison Table (Liu et al. Tables 4–5)

Aggregates LLM extraction vs regex and spaCy NER across all processed profiles.
Reproduce the paper's key finding: *LLM outperforms traditional methods in almost all scenarios.*

In [29]:
# ── Run baselines across all processed profiles ──────────────────────────
# Reads the HTML files already processed by the main pipeline.
# Uses the same extraction mode (preprocessed vs raw HTML) as the main pipeline
baseline_rows = []
for hf in tqdm(html_files, desc="Baselines"):
    html = hf.read_text(encoding="utf-8", errors="ignore")
    text = extract_readable_text(html)
    regex_r = regex_extract(text)
    spacy_r = spacy_extract(text)
    baseline_rows.append({
        "senator_id":             hf.stem,
        "regex_name":             regex_r["full_name"],
        "regex_email_found":      1 if regex_r["email"] else 0,
        "regex_phone_found":      1 if regex_r["phone"] else 0,
        "spacy_top_person":       spacy_r["persons_detected"][0] if spacy_r["persons_detected"] else None,
        "spacy_orgs_count":       len(spacy_r["orgs_detected"]),
    })

df_bl = pd.DataFrame(baseline_rows)
df_bl.to_csv(OUTPUT_DIR / "baselines.csv", index=False)

# ── Compare coverage: LLM vs baselines ───────────────────────────────────
df_t1  = pd.read_csv(OUTPUT_DIR / "task1_pii.csv")
merged_bl = df_t1.merge(df_bl, on="senator_id")

llm_name_cov   = merged_bl["full_name"].notna().mean()
regex_name_cov = merged_bl["regex_name"].notna().mean()
spacy_name_cov = merged_bl["spacy_top_person"].notna().mean()

llm_relig_cov   = merged_bl["religious_affiliation"].notna().mean()

print("=== Name extraction coverage (Liu et al. Table 4–5 analog) ===")
print(f"  LLM:   {llm_name_cov:.1%}")
print(f"  Regex: {regex_name_cov:.1%}")
print(f"  spaCy: {spacy_name_cov:.1%}")
print("\n=== Religious Affiliation extraction coverage ===")
print(f"  LLM:   {llm_relig_cov:.1%}")
print("\nFull baseline results saved to baselines.csv")

Baselines:   0%|          | 0/101 [00:00<?, ?it/s]

=== Name extraction coverage (Liu et al. Table 4–5 analog) ===
  LLM:   71.9%
  Regex: 91.1%
  spaCy: 92.1%

=== Religious Affiliation extraction coverage ===
  LLM:   49.8%

Full baseline results saved to baselines.csv


## 9a. Prompt Ablation Study (Liu et al. Table 13)

A rigorous comparison of all three prompt styles on a **fixed held-out subset of 25 senators**.
Each senator is extracted using direct, pseudocode, and ICL prompts.
This replicates Liu et al.'s prompt-style ablation (Section 4.2, Table 13) to quantify the effect of prompt engineering
on field coverage and accuracy.

**Features:**
- Fixed random seed (42) for reproducibility
- Separate results file (`ablation_results.json`) to keep ablation independent from main pipeline
- Resume-safe — skips already-completed combinations
- 3-second rate limit protection between API calls
- Coverage analysis across all 9 fields per prompt style


In [35]:
# ── Create fixed ablation subset (seed=42 for reproducibility) ────────────
# Use a held-out subset of 25 senators for rigorous prompt comparison
random.seed(42)
ablation_size = 25

# Sample ablation senators from all available files
ablation_senators = random.sample([f.stem for f in html_files], min(ablation_size, len(html_files)))

print(f"🔬 ABLATION SUBSET")
print(f"  Fixed seed:      42")
print(f"  Ablation size:   {len(ablation_senators)}")
print(f"  Senators:\n    {', '.join(ablation_senators[:5])}... (+{len(ablation_senators)-5} more)")
print()


🔬 ABLATION SUBSET
  Fixed seed:      42
  Ablation size:   25
  Senators:
    Roger_Marshall_KS, Catherine_Cortez_Masto_NV, Amy_Klobuchar_MN, Tim_Kaine_VA, James_Lankford_OK... (+20 more)



In [36]:
# ── Define all prompt styles for ablation ───────────────────────────────
ABLATION_STYLES = {
    "direct": TASK1_DIRECT,
    "pseudocode": TASK1_PSEUDOCODE,
    "icl": TASK1_ICL
}

# ── Load or initialize ablation results ──────────────────────────────────
ablation_path = OUTPUT_DIR / "ablation_results.json"

if ablation_path.exists():
    with open(ablation_path) as f:
        ablation_results = json.load(f)
    completed = set()
    for sid, styles_dict in ablation_results.items():
        for style in styles_dict.keys():
            completed.add((sid, style))
    print(f"✓ Resuming — {len(completed)} senator-style combinations already processed")
else:
    ablation_results = {}
    completed = set()

# ── Run ablation loop: all 3 prompts on all ablation senators ────────────
ablation_tasks = [(sid, style) for sid in ablation_senators for style in ABLATION_STYLES.keys()]
remaining_tasks = [t for t in ablation_tasks if t not in completed]

print(f"Remaining ablation tasks: {len(remaining_tasks)}/{len(ablation_tasks)}")
print()

for senator_id, style_name in tqdm(remaining_tasks, desc="Ablation loop"):
    # Load HTML and extract text
    html_file = [f for f in html_files if f.stem == senator_id]
    if not html_file:
        continue
    
    html = html_file[0].read_text(encoding="utf-8", errors="ignore")
    text = extract_readable_text(html)
    
    # Extract with current style
    prompt = ABLATION_STYLES[style_name]
    result = call_groq(prompt, text)
    
    # Initialize or update senator's results
    if senator_id not in ablation_results:
        ablation_results[senator_id] = {}
    
    ablation_results[senator_id][style_name] = result
    
    # Save incrementally (resume-safe)
    with open(ablation_path, "w") as f:
        json.dump(ablation_results, f, indent=2)
    
    # Rate limit protection — 3 seconds between calls
    time.sleep(3)

print(f"\n✓ Ablation complete. Results saved to: ablation_results.json")


Remaining ablation tasks: 75/75



Ablation loop:   0%|          | 0/75 [00:00<?, ?it/s]


✓ Ablation complete. Results saved to: ablation_results.json


In [37]:
# ── Flatten ablation results to CSV ──────────────────────────────────────
# Load ablation results
with open(OUTPUT_DIR / "ablation_results.json") as f:
    ablation_results = json.load(f)

# Define all fields to extract
T1_FIELDS_ABLATION = [
    "full_name",
    "birthdate",
    "birth_year_inferred",
    "gender",
    "race_ethnicity",
    "education",
    "committee_roles",
    "religious_affiliation",
    "religious_affiliation_inferred"
]

# Flatten to CSV: one row per (senator, prompt_style) combination
ablation_rows = []
for senator_id, styles_dict in ablation_results.items():
    for style_name, extraction_result in styles_dict.items():
        row = {
            "senator_id": senator_id,
            "prompt_style": style_name
        }
        
        # Check for extraction error
        if "error" in extraction_result:
            row["extraction_error"] = extraction_result.get("error")
            for field in T1_FIELDS_ABLATION:
                row[field] = None
        else:
            for field in T1_FIELDS_ABLATION:
                row[field] = extraction_result.get(field)
        
        ablation_rows.append(row)

df_ablation = pd.DataFrame(ablation_rows)
df_ablation.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)

print(f"✓ Flattened {len(df_ablation)} results to ablation_results.csv")
print(f"  Shape: {df_ablation.shape}")
print(f"\nPrompt style breakdown:")
print(df_ablation["prompt_style"].value_counts().to_string())


✓ Flattened 75 results to ablation_results.csv
  Shape: (75, 11)

Prompt style breakdown:
prompt_style
direct        25
pseudocode    25
icl           25


In [ ]:
# ── Coverage comparison: field coverage per prompt style ─────────────────
print("\n" + "=" * 80)
print("📊 PROMPT ABLATION — FIELD COVERAGE COMPARISON")
print("=" * 80)
print(f"\nAblation subset size: {len(ablation_senators)} senators")
print(f"Ablation styles: {', '.join(ABLATION_STYLES.keys())}")
print()

# Calculate coverage for each field by prompt style
coverage_by_style = {}
for style in ABLATION_STYLES.keys():
    style_data = df_ablation[df_ablation["prompt_style"] == style].copy()

    error_mask = style_data.get("extraction_error", pd.Series(dtype=str)).notna()
    valid_data = style_data[~error_mask]
    n_errors = int(error_mask.sum())
    n_valid = len(valid_data)

    if n_valid == 0:
        print(f"  ⚠ {style}: all {n_errors} results are errors — delete ablation_results.json and rerun")
        continue

    coverage_by_style[style] = {}
    for field in T1_FIELDS_ABLATION:
        filled = int(valid_data[field].notna().sum())
        pct = (filled / n_valid * 100)
        coverage_by_style[style][field] = {"filled": filled, "total": n_valid, "pct": pct}

if not coverage_by_style:
    print("⚠ No valid results to display.")
else:
    # Print coverage table
    print(f"{'Field':<35} | {'DIRECT':^15} | {'PSEUDOCODE':^15} | {'ICL':^15}")
    print("-" * 80)
    for field in T1_FIELDS_ABLATION:
        row_str = f"{field:<35} | "
        for style in ["direct", "pseudocode", "icl"]:
            if style in coverage_by_style:
                c = coverage_by_style[style][field]
                row_str += f"{c['filled']}/{c['total']} ({c['pct']:5.1f}%) | "
            else:
                row_str += f"{'N/A':^15} | "
        print(row_str)
    print("=" * 80)

    # Show average coverage per style
    print("\n📈 OVERALL COVERAGE BY PROMPT STYLE:")
    for style in ["direct", "pseudocode", "icl"]:
        if style in coverage_by_style:
            avg_cov = sum(coverage_by_style[style][f]["pct"] for f in T1_FIELDS_ABLATION) / len(T1_FIELDS_ABLATION)
            print(f"  {style:<15}: {avg_cov:6.1f}% (avg across all fields)")
        else:
            print(f"  {style:<15}: N/A (all errors)")

    print("\n💡 KEY OBSERVATIONS:")
    print("  • Compare direct vs pseudocode: Liu et al. find pseudocode slightly better for structured fields")
    print("  • Compare direct/pseudocode vs ICL: Liu et al. find ICL best for occupation-type fields")
    print("    (committee_roles, education, religious_affiliation)")
    print("=" * 80 + "\n")

In [38]:
import json
with open("../outputs/senate_results/ablation_results.json") as f:
    data = json.load(f)

# Check one senator across all 3 styles
first = list(data.keys())[0]
print(first)
print(json.dumps(data[first], indent=2))

Roger_Marshall_KS
{
  "direct": {
    "full_name": "Roger Marshall",
    "birthdate": null,
    "birth_year_inferred": null,
    "gender": null,
    "race_ethnicity": null,
    "education": [
      {
        "degree": "Bachelor\u2019s",
        "institution": "Kansas State University",
        "year": null
      },
      {
        "degree": "Medical Doctorate",
        "institution": "University of Kansas",
        "year": null
      }
    ],
    "committee_roles": [
      "Committee on Agriculture, Nutrition, and Forestry",
      "Chairman of the Subcommittee on Conservation, Forestry, Natural Resources, and Biotechnology",
      "Subcommittee on Food and Nutrition, Specialty Crops, Organics, and Research",
      "Committee on Finance",
      "Committee on Health, Education, Labor, and Pensions",
      "Chairman of the Subcommittee on Primary Health and Retirement Security",
      "Subcommittee on Employment and Workplace Safety",
      "Committee on the Budget"
    ],
    "religious_

## 14. Next Steps

### Quick Task Checklist

| Task | Section | Priority | Time | Status |
|------|---------|----------|------|--------|
| Full pipeline run (all 100 senators) | 5 | ✅ High | 2-4 hrs | [x] Done |
| Ground truth creation & prep | Manual | ✅ High | 4-6 hrs | [ ] Pending |
| Evaluation vs ground truth | 8a | High | 30 min | [ ] Needs GT |
| Prompt variant testing | 5→6 | Medium | 1-2 hrs | [ ] Next |
| Model scaling (8B vs 70B) | 5b | Medium | 3-5 hrs | [ ] Optional |
| Ablation study (controlled) | 9a | Medium | 2-3 hrs | [ ] Optional |
| Error diagnosis | 9b | Low | 1-2 hrs | [ ] Debug only |
| Religion signal annotation | 8b | ✅ High | 1-2 hrs | [ ] Pending |

---

### 1. Main Pipeline: Test All Prompt Styles (Section 5)

Compare how different prompting formats affect coverage across all 100 senators.

**Steps:**
```python
# Section 2b (Configuration):
RUN_ALL_PROMPT_STYLES = True
STYLES_TO_RUN = ["direct", "pseudocode", "icl"]

# Then re-run Section 2b (updates session metadata)
# Then re-run Section 5 (main pipeline)
# Then run Section 6 (coverage analysis)
```

**Output:**
- `results_raw.json`: 3× entries per senator (one per style)
- `task1_pii.csv`: Coverage stats for all styles
- **Expected finding:** Validate Liu et al. — which style wins for each field type?

---

### 2. Main Pipeline: Single Prompt Style (Faster)

For quick iteration or debugging a specific style.

**Steps:**
```python
# Section 2b (Configuration):
RUN_ALL_PROMPT_STYLES = False
ACTIVE_PROMPT_STYLE = "pseudocode"  # or "icl" or "direct"

# Then re-run Section 2b
# Then re-run Section 5
```

**Use when:**
- Testing a new prompt template variant
- Debugging API issues with one style
- Budget-conscious (1/3 API cost)

---

### 3. Ground Truth Collection (Manual, ~4-6 hours)

Create manually-labeled reference dataset for rigorous evaluation.

**Steps:**
1. Select 10-25 senators from `outputs/senate_results/task1_pii.csv`
2. Verify info from [Ballotpedia](https://ballotpedia.org/) or official Senate bios
3. Create `outputs/senate_results/ground_truth.csv`:
   ```csv
   senator_id,full_name,birthdate,gender,race_ethnicity,education,committee_roles,religious_affiliation
   0,"John Smith","1965-03-15","Male",,"{BA: State U. 1987}","Education; Armed Services","Christian"
   1,"Jane Doe","1958-07-22","Female","Asian-American","{BS: Stanford; JD: Yale}","Healthcare","Buddhist"
   ```
4. Once created, re-run **Section 8a** to auto-evaluate

**Output:** `accuracy.csv` with Accuracy, Rouge-1, BERT Score per field

---

### 4. Ablation Study: Controlled Comparison (Section 9a)

Run all 3 styles on same **fixed 25-senator subset** (seed=42) for fair comparison.

**Why ablation?**
- Controls senator variance
- Fair comparison (same input → different outputs)
- Separates "better prompt" from "easier senator"

**Steps:**
```python
# Section 9a (already auto-configured):
ABLATION_SEED = 42           # Ensures same 25 senators
ABLATION_SUBSET_SIZE = 25
STYLES_TO_RUN = ["direct", "pseudocode", "icl"]

# Re-run all cells in Section 9a
```

**Output:**
- `ablation_results.csv`: One row per (senator, style) combo
- **Expected:** ICL > Pseudocode > Direct (validate Liu et al. Table 2)

---

### 5. Model Scaling: 8B vs 70B Comparison (Section 5b)

Quantify the impact of model size on field coverage.

**Steps:**
```python
# Section 5b (Model Comparison):
MODEL_PAIRS = [("llama-3.1-8b-instant", "llama-3.3-70b-versatile")]
SUBSET_SENATORS = 20  # Limit for rate limit safety

# Re-run Section 5b
```

**Output:** `model_comparison.csv`

**Expected (Liu et al. Table 3):**
- 70B > 8B on semantic fields (religion, committees)
- Similar on explicit fields (name, birthdate)
- 70B costs ~5× more but higher quality inference

---

### 6. Error Analysis: Diagnose Failures (Section 9b)

Identify why specific extractions fail or are inconsistent.

**Error categories:**
- JSON parsing failures (malformed LLM output)
- Missing fields (LLM returned null when data present)
- Structural errors (education as string vs list)
- Inference false positives (religious_affiliation_inferred=true but actually missing)

**Steps:**
```python
# Section 9b (Error Analysis):
ERROR_THRESHOLD = 0.3  # Show senators with >30% field failures

# Re-run Section 9b
```

**Output:** `error_report.json` with examples

**Use when:**
- Spot-checking results for quality
- Debugging prompt template issues
- Identifying senator-level patterns (e.g., very long bios fail more?)

---

### 7. Baseline Comparison: LLM vs Traditional NLP (Section 8b)

Compare against Regex + spaCy NER baselines.

**Already completed** — shows:
- LLM ~60-85% coverage on explicit fields
- Regex/spaCy ~30-40% (limited, no inference)
- **LLM-only capabilities:** religious_affiliation inference

**To re-run:**
```python
# Section 8b:
# Just re-run all cells (compares outputs already in memory)
```

---

## Session Metadata & Resumption

Every run saves checkpoint info to `outputs/senate_results/session_metadata.json`:

```json
{
  "session_timestamp": "2026-04-01T15:32:45.123456",
  "active_prompt_style": "direct",
  "active_model": "llama-3.1-8b-instant",
  "run_all_prompt_styles": false,
  "last_senator_index": 45,
  "execution_status": "in_progress"
}
```

**To resume:** Just re-run Section 5 — it auto-detects the checkpoint and continues.

---

## Recommended Workflow

**Week 1: Baseline**
- [x] Complete Section 5 (full pipeline)
- [ ] Run Section 6 (coverage report)
- [ ] Create ground_truth.csv (10-15 senators)
- [ ] Run Section 8a (initial eval)
- [ ] Run Section 8b (religion signal annotation)

**Week 2: Prompt Comparison**
- [ ] Re-run with `RUN_ALL_PROMPT_STYLES = True`
- [ ] Run Section 9a (ablation study)
- [ ] Compare results vs Liu et al. Table 2

**Week 3: Model & Deep Dive**
- [ ] Run Section 5b (8B vs 70B)
- [ ] Run Section 9b (error analysis)
- [ ] Expand ground_truth.csv (25-50 senators)

**Week 4: Synthesis**
- [ ] Compile comparison table
- [ ] Generate visualizations (Section 7)
- [ ] Write findings report

---

## Troubleshooting

| Issue | Solution |
|-------|----------|
| `GROQ_API_KEY not set` | `export GROQ_API_KEY="sk_..."` before notebook |
| HTTP 429 (rate limit) | Wait 2 min; Section 5 auto-resumes from checkpoint |
| JSON parsing error | Run Section 9b to identify malformed entries |
| ground_truth.csv not found | Create CSV in `outputs/senate_results/` with proper columns |
| Empty results_raw.json | Check API key, re-run Section 5 |
| Inconsistent extractions | Use Section 9a (ablation) for same-senator comparison |

---

## Key Configuration Variables

```python
# Section 2b: Prompt & Model Selection
RUN_ALL_PROMPT_STYLES      # True = all 3 styles; False = single style
ACTIVE_PROMPT_STYLE        # "direct", "pseudocode", or "icl"
MODEL_OVERRIDE             # "llama-3.1-8b-instant" or "llama-3.3-70b-versatile"

# Section 9a: Ablation Control
ABLATION_SEED              # 42 (fixed for reproducibility)
ABLATION_SUBSET_SIZE       # 25 (controlled senator set)

# Section 9b: Error Detection
ERROR_THRESHOLD            # 0.3 (show senators with >30% failures)

# Section 5b: Model Comparison
MODEL_PAIRS                # Which model sizes to compare
SUBSET_SENATORS            # How many to test (for rate limit safety)
```

---

## Citation

> Liu et al., *Auditing LLM Agents for Information Disclosure: Are Your Extracted Personal Profiles Accurate?* USENIX Security 2025.

This notebook replicates Liu et al. Sections 4.1–4.2 (Task 1 PII extraction with 3 prompt styles) and extends with ablation, error analysis, and session tracking for reproducibility.